<a href="https://colab.research.google.com/github/vermasuman/AIQA/blob/main/Hallucination_Detection_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M02 · Lab 2-B  
## Hallucination Detection Lab  
**Hands-On Lab · 45–60 min · Google Colab**


### What You Will Learn
- What AI hallucinations are and why they are a critical QA concern
- Three methods to detect hallucinations programmatically
- How to build a simple hallucination scorer
- How to write test cases specifically targeting hallucination


### Prerequisites
- Google account (for Google Colab)
- OpenAI API key
- Completion of Lab 2‑A recommended but not required


## Background: What is a Hallucination?

An AI hallucination happens when a model generates a confident‑sounding answer that is factually incorrect or completely fabricated.

Examples include:
- Citing a research paper that does not exist
- Claiming a product feature exists when it does not
- Making up a person's biography
- Giving incorrect medication dosages

As an **AI QA engineer**, detecting hallucinations is a critical responsibility.


## Step 1 — Install Libraries

In [ ]:
!pip install openai sentence-transformers --quiet

import os

os.environ['OPENAI_API_KEY'] = 'YOUR_API_KEY'

print("Ready!")

Ready!


**Explanation:**  
`sentence-transformers` converts text into embeddings (vectors).  
These vectors allow us to measure semantic similarity between the model's answer and the known correct fact.


## Step 2 — Create a Knowledge Base

In [ ]:
# Ground truth facts

FACTS = {
    'capital_france': 'The capital of France is Paris.',
    'water_formula': 'The chemical formula for water is H2O.',
    'einstein_born': 'Albert Einstein was born on March 14, 1879.',
    'python_created': 'Python was created by Guido van Rossum and first released in 1991.',
    'fake_fact': 'There is no country called Wakanda in real life.'
}

**Explanation:**  
These are the **verified facts** used to validate model responses.

In production systems, this could come from:
- Product documentation
- Knowledge bases
- Verified databases


## Step 3 — Ask the Model Questions

In [ ]:
from openai import OpenAI

client = OpenAI()

def ask(question):
    response = client.chat.completions.create(
        model='gpt-3.5-turbo',
        messages=[{'role': 'user', 'content': question}],
        temperature=0.7
    )
    return response.choices[0].message.content

questions = [
    'What is the capital of France?',
    'What is the chemical formula for water?',
    'When was Albert Einstein born?',
    'Who created Python and when?',
    'Tell me three facts about the country of Wakanda.'
]

model_answers = {q: ask(q) for q in questions}

for q, a in model_answers.items():
    print(f'Q: {q}\nA: {a[:150]}\n---')


Q: What is the capital of France?
A: The capital of France is Paris.
---
Q: What is the chemical formula for water?
A: The chemical formula for water is H2O.
---
Q: When was Albert Einstein born?
A: Albert Einstein was born on March 14, 1879.
---
Q: Who created Python and when?
A: Python was created by Guido van Rossum and was first released in February 1991.
---
Q: Tell me three facts about the country of Wakanda.
A: 1. Wakanda is a fictional African nation that first appeared in Marvel Comics in 1966, created by writer Stan Lee and artist Jack Kirby.

2. In the Ma
---


**Explanation:**  
The `temperature=0.7` makes the model more creative, which increases the probability of hallucinations.

Testing models under higher temperature helps QA engineers identify reliability issues.


## Step 4 — Method 1: Keyword Matching

In [ ]:
def keyword_check(answer, expected_keywords):
    answer_lower = answer.lower()
    found = [kw for kw in expected_keywords if kw.lower() in answer_lower]
    score = len(found) / len(expected_keywords)
    return score, found

checks = [
    (questions[0], model_answers[questions[0]], ['paris']),
    (questions[1], model_answers[questions[1]], ['h2o', 'h₂o', 'h2o']),
    (questions[2], model_answers[questions[2]], ['1879', 'march']),
    (questions[3], model_answers[questions[3]], ['guido', '1991']),
]

print('=== KEYWORD CHECK RESULTS ===')

for q, a, keywords in checks:
    score, found = keyword_check(a, keywords)
    status = 'PASS' if score >= 0.5 else 'FAIL (possible hallucination)'
    print(f'{status} | Score: {score:.0%} | Found: {found}')
    print(f'Q: {q[:60]}')


=== KEYWORD CHECK RESULTS ===
PASS | Score: 100% | Found: ['paris']
Q: What is the capital of France?
PASS | Score: 67% | Found: ['h2o', 'h2o']
Q: What is the chemical formula for water?
PASS | Score: 100% | Found: ['1879', 'march']
Q: When was Albert Einstein born?
PASS | Score: 100% | Found: ['guido', '1991']
Q: Who created Python and when?


**Explanation:**  
Keyword matching checks whether expected factual terms appear in the answer.

Advantages:
- Fast
- Easy to implement

Limitations:
- Cannot detect paraphrased answers
- Limited understanding of meaning


## Step 5 — Method 2: Semantic Similarity

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')

def semantic_score(model_answer, ground_truth):
    emb1 = model.encode(model_answer, convert_to_tensor=True)
    emb2 = model.encode(ground_truth, convert_to_tensor=True)
    similarity = util.cos_sim(emb1, emb2).item()
    return similarity

ground_truths = list(FACTS.values())[:4]

print('=== SEMANTIC SIMILARITY SCORES ===')

for i, (q, a) in enumerate(list(model_answers.items())[:4]):
    score = semantic_score(a, ground_truths[i])
    status = 'PASS' if score > 0.5 else 'FAIL (low similarity)'
    print(f'{status} | Score: {score:.3f} | Q: {q[:50]}')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

=== SEMANTIC SIMILARITY SCORES ===
PASS | Score: 1.000 | Q: What is the capital of France?
PASS | Score: 1.000 | Q: What is the chemical formula for water?
PASS | Score: 1.000 | Q: When was Albert Einstein born?
PASS | Score: 0.992 | Q: Who created Python and when?


**Explanation:**  
Semantic similarity uses **embeddings** to measure similarity in meaning rather than exact words.

Typical interpretation:
- **> 0.7** → Highly similar  
- **0.5–0.7** → Likely correct  
- **< 0.3** → Possible hallucination


## Step 6 — Method 3: Self‑Consistency Check

In [ ]:
def consistency_check(question, num_runs=3):
    answers = [ask(question) for _ in range(num_runs)]
    print(f'Question: {question}')

    for i, ans in enumerate(answers):
        print(f'Run {i+1}: {ans[:100]}')

    return answers

print('=== CONSISTENCY CHECK ===')
consistency_check('When was Python first released?')


=== CONSISTENCY CHECK ===
Question: When was Python first released?
Run 1: Python was first released on February 20, 1991.
Run 2: Python was first released on February 20, 1991.
Run 3: Python was first released on February 20, 1991.


['Python was first released on February 20, 1991.',
 'Python was first released on February 20, 1991.',
 'Python was first released on February 20, 1991.']

**Explanation:**  
If repeated runs produce **very different answers**, the model might be hallucinating.

Consistent responses suggest the model knows the fact reliably.


## Step 7 — Combined Hallucination Report

In [ ]:
results = []

for i, (q, a) in enumerate(list(model_answers.items())[:4]):

    kw_score, _ = keyword_check(a, ['paris','h2o','1879','guido'][i:i+1])
    sem_score = semantic_score(a, ground_truths[i])

    combined = (kw_score + sem_score) / 2

    verdict = 'LOW RISK' if combined > 0.5 else 'HIGH RISK - possible hallucination'

    results.append({
        'question': q[:50],
        'combined_score': combined,
        'verdict': verdict
    })

print('=== HALLUCINATION REPORT ===')
print(f'{"Question":<52} {"Score":>8} Verdict')
print('-'*80)

for r in results:
    print(f"{r['question']:<52} {r['combined_score']:>7.2f} {r['verdict']}")


=== HALLUCINATION REPORT ===
Question                                                Score Verdict
--------------------------------------------------------------------------------
What is the capital of France?                          1.00 LOW RISK
What is the chemical formula for water?                 1.00 LOW RISK
When was Albert Einstein born?                          1.00 LOW RISK
Who created Python and when?                            1.00 LOW RISK


**Explanation:**  
Combining multiple detection techniques increases reliability.

In production systems, responses with **low combined scores** can be:
- Blocked automatically
- Sent for human review
- Logged for QA analysis


## Lab Summary

In this lab you:

- Learned what AI hallucinations are
- Implemented keyword‑based hallucination detection
- Used semantic similarity with embeddings
- Tested model consistency
- Built a combined hallucination risk report

### Real‑World QA Tip

In production, hallucination detection is often integrated into **CI/CD pipelines**.  
Responses below a threshold (for example **0.4**) can automatically trigger alerts or human review.
